# 05F — Final Full-Data Fit and Primary Model Serialization

**Build ID:** `05F_V1_FINAL_FULL_DATA_FIT_MODEL_SERIALIZATION`

The primary pipeline is frozen as the exact fixed 05C row-weighted `E_FULL_EXTENDED` LightGBM model with raw `P(Hold=1)` output.

This stage fits that frozen model once on all 304,143 engaged units, serializes it, verifies reload equivalence, and exports descriptive feature importance. No tuning, calibration, or threshold selection occurs. Scientific performance remains the incident-held-out OOF evaluation.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05F_V1_FINAL_FULL_DATA_FIT_MODEL_SERIALIZATION"

from pathlib import Path
import sys, subprocess, importlib.util, gc, json, hashlib, time

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
OOF_PRIMARY_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "final_primary_model_v1"
MODEL_ROOT = PROJECT_ROOT / "models" / "fireline_hold_primary_v1"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, FEATURE_SETS_PATH, OOF_PRIMARY_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
EXPECTED_FEATURES = 803

PRIMARY_MODEL_ID = "FIRELINE_HOLD_PRIMARY_V1"
FEATURE_SET_NAME = "E_FULL_EXTENDED"
MODEL_SEED = 20260825

FROZEN_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
    "random_state": MODEL_SEED,
}

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "lightgbm": "lightgbm",
    "joblib": "joblib",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import joblib
from lightgbm import LGBMClassifier

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print(FROZEN_PARAMS)

## 1. Load frozen feature set and full modeling cohort

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    FEATURE_SETS = json.load(f)

features = FEATURE_SETS[FEATURE_SET_NAME]

assert len(features) == EXPECTED_FEATURES
assert len(features) == len(set(features))

data = pd.read_parquet(
    MASTER_PATH,
    columns=[
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "hold_label",
        *features,
    ],
)

assert len(data) == EXPECTED_UNITS
assert data["unit_uid"].is_unique
assert data["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert data["hold_label"].notna().all()
assert set(data["hold_label"].astype(int).unique()) == {0, 1}

X = data[features].to_numpy(
    dtype=np.float32,
    na_value=np.nan,
)

y = data["hold_label"].to_numpy(dtype=np.int8)

assert X.shape == (EXPECTED_UNITS, EXPECTED_FEATURES)

print("Training matrix:", X.shape, X.dtype)
print("Approx GiB:", f"{X.nbytes/1024**3:.3f}")
print("Held prevalence:", float(y.mean()))

## 2. Fit final frozen model

One full-data fit only. No validation set, early stopping, tuning, calibration, or threshold selection.

In [ ]:
model = LGBMClassifier(**FROZEN_PARAMS)

t0 = time.time()
model.fit(X, y)
fit_seconds = time.time() - t0

assert model.n_estimators_ == 300

print(f"Fit complete in {fit_seconds/60:.3f} minutes")
print("Trees:", model.n_estimators_)

## 3. Serialize model artifacts

In [ ]:
NATIVE_MODEL_PATH = MODEL_ROOT / "fireline_hold_primary_v1.txt"
JOBLIB_MODEL_PATH = MODEL_ROOT / "fireline_hold_primary_v1.joblib"
FEATURE_LIST_PATH = MODEL_ROOT / "fireline_hold_primary_v1_features.json"
PARAMS_PATH = MODEL_ROOT / "fireline_hold_primary_v1_params.json"

model.booster_.save_model(str(NATIVE_MODEL_PATH))
joblib.dump(model, JOBLIB_MODEL_PATH, compress=3)

with FEATURE_LIST_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "model_id": PRIMARY_MODEL_ID,
            "feature_set": FEATURE_SET_NAME,
            "feature_count": len(features),
            "features": features,
        },
        f,
        indent=2,
    )

with PARAMS_PATH.open("w", encoding="utf-8") as f:
    json.dump(FROZEN_PARAMS, f, indent=2)

for p in [NATIVE_MODEL_PATH, JOBLIB_MODEL_PATH, FEATURE_LIST_PATH, PARAMS_PATH]:
    assert p.exists(), p

print("Saved:", NATIVE_MODEL_PATH)
print("Saved:", JOBLIB_MODEL_PATH)

## 4. Reload equivalence smoke test

In [ ]:
SMOKE_N = 2000
rng = np.random.default_rng(MODEL_SEED)

smoke_idx = np.sort(
    rng.choice(
        EXPECTED_UNITS,
        size=SMOKE_N,
        replace=False,
    )
)

X_smoke = X[smoke_idx]

pred_memory = model.predict_proba(X_smoke)[:, 1]

model_joblib = joblib.load(JOBLIB_MODEL_PATH)
pred_joblib = model_joblib.predict_proba(X_smoke)[:, 1]

model_native = lgb.Booster(model_file=str(NATIVE_MODEL_PATH))
pred_native = model_native.predict(X_smoke)

max_abs_joblib = float(
    np.max(np.abs(pred_memory - pred_joblib))
)

max_abs_native = float(
    np.max(np.abs(pred_memory - pred_native))
)

serialization_audit = pd.DataFrame([
    {
        "comparison": "memory_vs_joblib",
        "max_abs_probability_difference": max_abs_joblib,
        "pass": max_abs_joblib < 1e-12,
    },
    {
        "comparison": "memory_vs_native",
        "max_abs_probability_difference": max_abs_native,
        "pass": max_abs_native < 1e-10,
    },
])

display(serialization_audit)
assert serialization_audit["pass"].all()

serialization_audit.to_csv(
    OUTPUT_ROOT / "MODEL_SERIALIZATION_AUDIT_V1.csv",
    index=False,
)

## 5. Descriptive full-data feature importance

These are model-structure summaries only; they are not causal effects or held-out performance.

In [ ]:
booster = model.booster_

importance = pd.DataFrame({
    "feature": features,
    "importance_split": booster.feature_importance(
        importance_type="split"
    ),
    "importance_gain": booster.feature_importance(
        importance_type="gain"
    ),
})

importance["importance_gain_fraction"] = (
    importance["importance_gain"]
    / importance["importance_gain"].sum()
)

importance["rank_gain"] = (
    importance["importance_gain"]
    .rank(method="min", ascending=False)
    .astype(int)
)

importance["rank_split"] = (
    importance["importance_split"]
    .rank(method="min", ascending=False)
    .astype(int)
)

importance = importance.sort_values(
    ["rank_gain", "feature"]
).reset_index(drop=True)

display(importance.head(40))

importance.to_csv(
    OUTPUT_ROOT / "FINAL_MODEL_FEATURE_IMPORTANCE_V1.csv",
    index=False,
)

## 6. Artifact manifest, model card, and final checks

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

artifact_manifest = pd.DataFrame([
    {
        "artifact": p.name,
        "path": str(p),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    }
    for p in [
        NATIVE_MODEL_PATH,
        JOBLIB_MODEL_PATH,
        FEATURE_LIST_PATH,
        PARAMS_PATH,
    ]
])

artifact_manifest.to_csv(
    OUTPUT_ROOT / "MODEL_ARTIFACT_MANIFEST_V1.csv",
    index=False,
)

model_card = {
    "build_id": NOTEBOOK_BUILD_ID,
    "model_id": PRIMARY_MODEL_ID,
    "source_selection": "05C_FIXED_ROW_RAW",
    "target": {
        "column": "hold_label",
        "positive_class": 1,
        "positive_meaning": "Held",
        "negative_class": 0,
        "negative_meaning": "Burned Over",
        "estimand": "P(Hold=1 | engaged fireline unit, X)",
    },
    "training_cohort": {
        "rows": EXPECTED_UNITS,
        "incidents": EXPECTED_INCIDENTS,
        "engaged_only": True,
    },
    "features": {
        "feature_set": FEATURE_SET_NAME,
        "count": EXPECTED_FEATURES,
        "ordered_feature_file": str(FEATURE_LIST_PATH),
        "missing_value_handling": "LightGBM native missing-value routing",
    },
    "model": {
        "family": "LightGBM LGBMClassifier",
        "parameters": FROZEN_PARAMS,
        "training_weighting": "ordinary row weighting",
        "posthoc_calibration": "none",
        "classification_threshold": None,
        "probability_output": "raw predict_proba probability for class 1",
    },
    "evaluation_contract": {
        "performance_source": "incident-held-out OOF results from 05C/05E/05E1",
        "full_data_fit_is_performance_estimate": False,
        "grouping_key": "_irwin_norm",
        "random_row_split_prohibited": True,
    },
    "notes": [
        "Full-data fit is a deployment artifact, not a performance estimate.",
        "Nested calibration was not adopted because pooled improvements were small/uncertain while incident-macro probability loss worsened.",
        "Incident-equal modeling remains a secondary robustness analysis.",
        "Full-data feature importance is descriptive and non-causal.",
        "LANDFIRE 2016 / 2020-capable fuel-state provenance remains a separate documentation caveat.",
    ],
}

MODEL_CARD_PATH = MODEL_ROOT / "fireline_hold_primary_v1_model_card.json"

with MODEL_CARD_PATH.open("w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2)

checks = pd.DataFrame([
    {
        "check": "exact_304143_training_units",
        "pass": len(data) == EXPECTED_UNITS,
    },
    {
        "check": "exact_381_training_incidents",
        "pass": data["_irwin_norm"].nunique() == EXPECTED_INCIDENTS,
    },
    {
        "check": "exact_803_frozen_features",
        "pass": len(features) == EXPECTED_FEATURES,
    },
    {
        "check": "exact_frozen_05c_parameter_contract",
        "pass": (
            FROZEN_PARAMS["n_estimators"] == 300
            and FROZEN_PARAMS["learning_rate"] == 0.05
            and FROZEN_PARAMS["num_leaves"] == 31
            and FROZEN_PARAMS["min_child_samples"] == 150
            and FROZEN_PARAMS["reg_lambda"] == 1.0
            and "subsample_freq" not in FROZEN_PARAMS
        ),
    },
    {
        "check": "ordinary_row_weighting_only",
        "pass": True,
    },
    {
        "check": "no_posthoc_calibration_in_primary_pipeline",
        "pass": True,
    },
    {
        "check": "no_hyperparameter_tuning_in_05f",
        "pass": True,
    },
    {
        "check": "no_threshold_selection_in_05f",
        "pass": True,
    },
    {
        "check": "native_model_serialized",
        "pass": NATIVE_MODEL_PATH.exists(),
    },
    {
        "check": "joblib_model_serialized",
        "pass": JOBLIB_MODEL_PATH.exists(),
    },
    {
        "check": "serialization_equivalence_passed",
        "pass": bool(serialization_audit["pass"].all()),
    },
    {
        "check": "feature_contract_serialized",
        "pass": FEATURE_LIST_PATH.exists(),
    },
    {
        "check": "model_card_serialized",
        "pass": MODEL_CARD_PATH.exists(),
    },
    {
        "check": "performance_claims_remain_oof_only",
        "pass": True,
    },
])

display(artifact_manifest)
display(checks)

PASS = bool(checks["pass"].all())

print()
print(
    "05F FINAL FULL-DATA FIT / SERIALIZATION:",
    (
        "PASS / PRIMARY MODEL ARTIFACT FROZEN"
        if PASS
        else "REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT / "FINAL_MODEL_SERIALIZATION_CHECKS_V1.csv",
    index=False,
)

## 7. Final report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = [
    "# 05F Final Full-Data Fit and Primary Model Serialization",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    "## Frozen primary model",
    "",
    f"`{PRIMARY_MODEL_ID}`",
    "",
    "Probability output: `raw LightGBM P(Hold=1)`",
    "",
    "Post-hoc calibration: `none`",
    "",
    f"Training units: **{EXPECTED_UNITS:,}**",
    f"Training incidents: **{EXPECTED_INCIDENTS}**",
    f"Features: **{EXPECTED_FEATURES}**",
    "",
    "## Frozen parameters",
    "",
    "```json",
    json.dumps(FROZEN_PARAMS, indent=2),
    "```",
    "",
    "## Fit runtime",
    "",
    f"{fit_seconds/60:.3f} minutes",
    "",
    "## Serialization audit",
    "",
    tomd(serialization_audit),
    "",
    "## Artifact manifest",
    "",
    tomd(artifact_manifest),
    "",
    "## Top 40 feature importance by gain",
    "",
    tomd(importance.head(40)),
    "",
    "## Final checks",
    "",
    tomd(checks),
    "",
    "## Decision",
    "",
    (
        "`PASS / PRIMARY MODEL ARTIFACT FROZEN`"
        if PASS
        else "`REVIEW REQUIRED`"
    ),
    "",
    "The full-data fit is a deployment artifact. Scientific performance remains the incident-held-out OOF evaluation.",
]

REPORT_PATH = OUTPUT_ROOT / "FINAL_PRIMARY_MODEL_REPORT_V1.md"

REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

print("\n".join(report_lines))
print()
print("Saved:", REPORT_PATH)